# OBO App Registration Setup (Minimal)

This notebook does one thing: configure and verify the two Entra app registrations for robust OBO with Copilot Studio custom connectors.

It expects these values in .env:
- TENANT_ID
- SERVICE_APP_ID
- CONNECTOR_CLIENT_ID
- AZURE_API_CONNECTIONS_SP_ID (optional, defaults to fe053c5f-3692-4f14-aef2-ee34fc081cae)

## Install Notebook Dependencies

This cell installs the Python packages used by the setup and verification steps.

In [ ]:
%pip install requests azure-identity python-dotenv
print("Dependencies installed.")

## Load Configuration and Graph Helpers

This cell loads `.env`, validates the required values, and prepares Azure CLI and Microsoft Graph helpers.

In [ ]:
import os
import json
import uuid
import shutil
import subprocess
from pathlib import Path

import requests
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

ENV_PATH = Path(".env")
if ENV_PATH.exists():
    load_dotenv(ENV_PATH, override=False)

TENANT_ID = (os.getenv("TENANT_ID") or "").strip()
SERVICE_APP_ID = (os.getenv("SERVICE_APP_ID") or "").strip()
CONNECTOR_CLIENT_ID = (os.getenv("CONNECTOR_CLIENT_ID") or "").strip()
AZURE_API_CONNECTIONS_SP_ID = (os.getenv("AZURE_API_CONNECTIONS_SP_ID") or "fe053c5f-3692-4f14-aef2-ee34fc081cae").strip()

if not TENANT_ID:
    raise RuntimeError("TENANT_ID is required in .env")
if not SERVICE_APP_ID:
    raise RuntimeError("SERVICE_APP_ID is required in .env")
if not CONNECTOR_CLIENT_ID:
    raise RuntimeError("CONNECTOR_CLIENT_ID is required in .env")

AZ_CLI = shutil.which("az") or shutil.which("az.cmd")
if not AZ_CLI:
    raise RuntimeError("Azure CLI (az) was not found in this notebook kernel environment")

def az_json(args):
    proc = subprocess.run([AZ_CLI] + args + ["-o", "json"], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError("az " + " ".join(args) + " failed: " + proc.stderr.strip())
    text = proc.stdout.strip()
    return json.loads(text) if text else {}

credential = AzureCliCredential()

def get_graph_headers():
    token = credential.get_token("https://graph.microsoft.com/.default").token
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def graph_patch_application(app_object_id, payload):
    url = f"https://graph.microsoft.com/v1.0/applications/{app_object_id}"
    resp = requests.patch(url, headers=get_graph_headers(), json=payload, timeout=60)
    if resp.status_code >= 300:
        body = resp.text
        if resp.status_code == 401 and "InteractionRequired" in body:
            raise RuntimeError(
                "Graph token requires reauthentication due to policy change. "
                "Run az login, then rerun this notebook cell. Details: " + body
            )
        raise RuntimeError(f"Graph patch failed ({resp.status_code}): {body}")

print("Loaded configuration and Graph auth.")

## Configure the Two App Registrations

This cell applies the Entra app registration changes for the service app and connector app.

In [ ]:
# Apply OBO app registration configuration
service_app = az_json(["ad", "app", "show", "--id", SERVICE_APP_ID])
connector_app = az_json(["ad", "app", "show", "--id", CONNECTOR_CLIENT_ID])

service_obj_id = service_app["id"]
connector_obj_id = connector_app["id"]
service_app_uri = f"api://{SERVICE_APP_ID}"
connector_app_uri = f"api://{CONNECTOR_CLIENT_ID}"

# Service app: set identifier URI and ensure access_as_user scope
service_scopes = service_app.get("api", {}).get("oauth2PermissionScopes", [])
service_scope = next((s for s in service_scopes if s.get("value") == "access_as_user"), None)
if not service_scope:
    service_scope = {
        "id": str(uuid.uuid4()),
        "value": "access_as_user",
        "type": "User",
        "isEnabled": True,
        "adminConsentDisplayName": "Allow connector to call service on behalf of users",
        "adminConsentDescription": "Allows delegated access to the MCP service on behalf of users",
        "userConsentDisplayName": "Allow connector to act on your behalf",
        "userConsentDescription": "Allows delegated access to the MCP service on your behalf"
    }
    service_scopes.append(service_scope)

graph_patch_application(service_obj_id, {
    "identifierUris": [service_app_uri],
    "api": {"oauth2PermissionScopes": service_scopes}
})

# Always resolve service scope ID from live app state (no assumptions)
service_app = az_json(["ad", "app", "show", "--id", SERVICE_APP_ID])
service_scope = next((s for s in service_app.get("api", {}).get("oauth2PermissionScopes", []) if s.get("value") == "access_as_user"), None)
if not service_scope:
    raise RuntimeError("Service app scope access_as_user was not created")
service_scope_id = service_scope["id"]

# Resolve Fabric delegated scope IDs from live SP state
fabric_sp = az_json(["ad", "sp", "show", "--id", "00000009-0000-0000-c000-000000000000"])
fabric_scope_ids = {}
for s in fabric_sp.get("oauth2PermissionScopes", []):
    if s.get("value") in ("DataAgent.Execute.All", "Item.Read.All"):
        fabric_scope_ids[s["value"]] = s["id"]
for needed in ("DataAgent.Execute.All", "Item.Read.All"):
    if needed not in fabric_scope_ids:
        raise RuntimeError(f"Missing Fabric scope: {needed}")

# Connector app: required resource access
required = connector_app.get("requiredResourceAccess", [])
required_other = [r for r in required if r.get("resourceAppId") not in (SERVICE_APP_ID, "00000009-0000-0000-c000-000000000000")]
required_service = {
    "resourceAppId": SERVICE_APP_ID,
    "resourceAccess": [{"id": service_scope_id, "type": "Scope"}]
}
required_fabric = {
    "resourceAppId": "00000009-0000-0000-c000-000000000000",
    "resourceAccess": [
        {"id": fabric_scope_ids["DataAgent.Execute.All"], "type": "Scope"},
        {"id": fabric_scope_ids["Item.Read.All"], "type": "Scope"}
    ]
}
merged_required = required_other + [required_service, required_fabric]

# Connector app: ensure access_as_user scope exists
connector_scopes = connector_app.get("api", {}).get("oauth2PermissionScopes", [])
connector_scope = next((s for s in connector_scopes if s.get("value") == "access_as_user"), None)
if not connector_scope:
    connector_scope = {
        "id": str(uuid.uuid4()),
        "value": "access_as_user",
        "type": "User",
        "isEnabled": True,
        "adminConsentDisplayName": "Allow Azure API Connections to obtain tokens on behalf of users",
        "adminConsentDescription": "Allows Azure API Connections to obtain tokens on behalf of the user",
        "userConsentDisplayName": "Allow connector to act on your behalf",
        "userConsentDescription": "Allows Azure API Connections to access resources on your behalf"
    }
    connector_scopes.append(connector_scope)

# Patch connector scopes and requiredResourceAccess first
graph_patch_application(connector_obj_id, {
    "identifierUris": [connector_app_uri],
    "requiredResourceAccess": merged_required,
    "api": {
        "oauth2PermissionScopes": connector_scopes
    }
})

# Re-read connector and resolve scope ID from live state before preauth
connector_app = az_json(["ad", "app", "show", "--id", CONNECTOR_CLIENT_ID])
connector_scope_live = next((s for s in connector_app.get("api", {}).get("oauth2PermissionScopes", []) if s.get("value") == "access_as_user"), None)
if not connector_scope_live:
    raise RuntimeError("Connector app scope access_as_user was not found after update")
connector_scope_live_id = connector_scope_live["id"]

# Patch preauthorization separately using looked-up scope GUID
preauth = [
    p for p in connector_app.get("api", {}).get("preAuthorizedApplications", [])
    if p.get("appId") != AZURE_API_CONNECTIONS_SP_ID
]
preauth.append({"appId": AZURE_API_CONNECTIONS_SP_ID, "delegatedPermissionIds": [connector_scope_live_id]})
graph_patch_application(connector_obj_id, {
    "api": {
        "preAuthorizedApplications": preauth
    }
})

# Admin consent
proc = subprocess.run([AZ_CLI, "ad", "app", "permission", "admin-consent", "--id", CONNECTOR_CLIENT_ID], capture_output=True, text=True)
if proc.returncode != 0:
    print("[WARN] Admin consent not confirmed by CLI. Review manually.")
    print(proc.stderr.strip())
else:
    print("[OK] Admin consent granted.")

print("[OK] OBO app registration configuration applied.")

## Verify the Live Entra State

This cell checks that the required app registration settings were applied successfully.

In [ ]:
# Verify final live state
service_verify = az_json(["ad", "app", "show", "--id", SERVICE_APP_ID])
connector_verify = az_json(["ad", "app", "show", "--id", CONNECTOR_CLIENT_ID])
fic_list = az_json(["ad", "app", "federated-credential", "list", "--id", CONNECTOR_CLIENT_ID])

service_uri_ok = service_verify.get("identifierUris", []) == [f"api://{SERVICE_APP_ID}"]
service_scope_ok = any(s.get("value") == "access_as_user" for s in service_verify.get("api", {}).get("oauth2PermissionScopes", []))
connector_uri_ok = connector_verify.get("identifierUris", []) == [f"api://{CONNECTOR_CLIENT_ID}"]
connector_scope_ok = any(s.get("value") == "access_as_user" for s in connector_verify.get("api", {}).get("oauth2PermissionScopes", []))
preauth_ok = any(p.get("appId") == AZURE_API_CONNECTIONS_SP_ID for p in connector_verify.get("api", {}).get("preAuthorizedApplications", []))

rra = connector_verify.get("requiredResourceAccess", [])
has_service_perm = any(r.get("resourceAppId") == SERVICE_APP_ID for r in rra)
has_fabric_perm = any(r.get("resourceAppId") == "00000009-0000-0000-c000-000000000000" for r in rra)

print("Verification results:")
print(f"  Service URI set               : {service_uri_ok}")
print(f"  Service scope access_as_user  : {service_scope_ok}")
print(f"  Connector URI set             : {connector_uri_ok}")
print(f"  Connector scope access_as_user: {connector_scope_ok}")
print(f"  Preauth Azure API Connections : {preauth_ok}")
print(f"  Delegated service permission  : {has_service_perm}")
print(f"  Delegated Fabric permissions  : {has_fabric_perm}")
print(f"  Federated credentials count   : {len(fic_list) if isinstance(fic_list, list) else 0}")

print("\nNext manual steps in Copilot Studio/Power Automate:")
print("1. Build custom connector from Swagger 2.0 and set Security with:")
print(f"   - Client ID: {CONNECTOR_CLIENT_ID}")
print("   - Resource URL: https://api.fabric.microsoft.com")
print("   - Scope: https://api.fabric.microsoft.com/.default")
print("   - Enable on-behalf-of login: true")
print("   - Secret option: Use managed identity")
print("2. Copy Redirect URL and managed identity Issuer/Subject from connector details.")
print("3. Add/update federated credential on connector app using those Issuer/Subject values.")
print("4. Recreate connection and test the connector in Power Platform.")
print("5. The service app remains part of the Entra OBO model, but the direct connector test uses Fabric as the runtime audience.")

## Generate Swagger 2.0 Template

This section creates a Swagger 2.0 file you can import into a Copilot Studio custom connector.

It keeps the two-app OBO setup in Entra, but uses the direct Fabric runtime audience that worked in connector testing:
- authorization URL and token URL from TENANT_ID
- runtime scope https://api.fabric.microsoft.com/.default
- MCP host and path from MCP_SERVER_URL

In [ ]:
from urllib.parse import urlparse

MCP_SERVER_URL = (os.getenv("MCP_SERVER_URL") or "https://api.fabric.microsoft.com/v1/mcp/workspaces/b0384965-ad16-45e4-818b-f6f792a9b14c/dataagents/8d01d144-69df-40a8-85dc-7b2574b57b89/agent").strip()
SWAGGER_OUTPUT_FILE = (os.getenv("SWAGGER_OUTPUT_FILE") or "workingswagger-obo-template.yaml").strip()

parsed = urlparse(MCP_SERVER_URL)
if parsed.scheme != "https":
    raise RuntimeError("MCP_SERVER_URL must use https")
if not parsed.netloc or not parsed.path:
    raise RuntimeError("MCP_SERVER_URL must include host and path")

tenant_authorize_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/authorize"
tenant_token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
runtime_scope = "https://api.fabric.microsoft.com/.default"

swagger_text = f"""swagger: '2.0'
info:
  title: Fabric MCP Server
  description: Swagger template for a Copilot Studio custom connector that calls a Fabric Data Agent MCP endpoint.
  version: 1.0.0
host: {parsed.netloc}
basePath: /
schemes:
  - https
paths:
  {parsed.path}:
    post:
      responses:
        '200':
          description: Immediate Response
      x-ms-agentic-protocol: mcp-streamable-1.0
      operationId: InvokeServer
      summary: Fabric MCP Server
      description: Invokes the MCP server endpoint for the configured Fabric Data Agent.
securityDefinitions:
  oauth2-auth:
    type: oauth2
    flow: accessCode
    authorizationUrl: {tenant_authorize_url}
    tokenUrl: {tenant_token_url}
    scopes:
      {runtime_scope}: {runtime_scope}
security:
  - oauth2-auth:
      - {runtime_scope}
"""

Path(SWAGGER_OUTPUT_FILE).write_text(swagger_text, encoding="utf-8")
print(f"[OK] Wrote Swagger template: {SWAGGER_OUTPUT_FILE}")
print(f"[INFO] Host: {parsed.netloc}")
print(f"[INFO] Path: {parsed.path}")
print(f"[INFO] Runtime scope: {runtime_scope}")
print("[INFO] This Swagger uses the direct Fabric audience that matched the successful connector test flow.")

## Connector Values -> Where To Enter Them

After the first connector Save in Power Platform Security, copy these values from the connector UI:
- Redirect URL
- Managed identity Issuer
- Managed identity Subject

Enter these values in Entra under the connector app registration (client app = CONNECTOR_CLIENT_ID):
- Authentication > Redirect URIs: add Redirect URL.
- Certificates and secrets > Federated credentials > Add credential:
  Scenario: Other issuer
  Issuer: Managed identity Issuer
  Subject: Managed identity Subject
  Audience: api://AzureADTokenExchange

Service app registration (resource app = SERVICE_APP_ID): no action required for this step.

In [ ]:
# Reconfigure checklist for Power Platform custom connector (Managed Identity + OBO app setup)
mcp_server_url = (os.getenv("MCP_SERVER_URL") or "").strip()
if not mcp_server_url and "MCP_SERVER_URL" in globals():
    mcp_server_url = str(globals()["MCP_SERVER_URL"]).strip()
if not mcp_server_url:
    mcp_server_url = "https://api.fabric.microsoft.com/v1/mcp/workspaces/<workspace-id>/dataagents/<dataagent-id>/agent"

scope_value = "https://api.fabric.microsoft.com/.default"
resource_url_value = "https://api.fabric.microsoft.com"

print("POWER PLATFORM CONNECTOR CONFIGURATION CHECKLIST")
print("")
print("Security tab values")
print("- Authentication type: OAuth 2.0")
print("- Identity Provider: Azure Active Directory")
print("- Secret options: Use managed identity")
print("- Enable on-behalf-of login: true")
print(f"- Client ID: {CONNECTOR_CLIENT_ID}")
print(f"- Tenant ID: {TENANT_ID} (not 'common')")
print(f"- Resource URL: {resource_url_value}")
print(f"- Scope: {scope_value}")
print("")
print("Definition tab values")
print("- Operation: InvokeServer")
print("- Verb: POST")
print(f"- URL: {mcp_server_url}")
print("- x-ms-agentic-protocol: mcp-streamable-1.0")
print("")
print("Copy these values from connector after first Save")
print("- Redirect URL")
print("- Managed identity Issuer")
print("- Managed identity Subject")
print("")
print("Enter those values in Entra here")
print(f"1. Connector app registration (client): {CONNECTOR_CLIENT_ID}")
print("   - Authentication > Redirect URIs: add Redirect URL")
print("   - Certificates and secrets > Federated credentials > Add credential")
print("     Scenario: Other issuer")
print("     Issuer: Managed identity Issuer")
print("     Subject: Managed identity Subject")
print("     Audience: api://AzureADTokenExchange")
print("")
print(f"2. Service app registration (resource): {SERVICE_APP_ID}")
print("   - No action in this step")
print("")
print("Final step")
print("- Save connector again, create a new connection, then test InvokeServer")
print("- This direct connector test uses Fabric as the runtime audience.")

## Test The Connector

Use the Power Platform Test tab to validate the connector with the same MCP request flow that worked here.

1. Save the connector after you finish the Security and Definition tabs.
2. Create a new connection for the connector. If you already created one with older settings, delete it and create a fresh connection.
3. Open the Test tab and choose the `InvokeServer` operation.
4. Turn on Raw Body.
5. Send the `initialize` request first.
6. After `initialize` succeeds, send `tools/list`.
7. After `tools/list` succeeds, send `tools/call` with one tool name returned by the server.

Expected behavior:
- `initialize` returns HTTP 200 with `jsonrpc: 2.0`, `result.protocolVersion`, and `result.serverInfo`.
- `tools/list` returns HTTP 200 with the tools exposed by the data agent.
- `tools/call` returns HTTP 200 with the tool result for the arguments you supplied.

Important: `InvokeServer` is the connector operation name in Power Platform. It is not the MCP `method` value inside the JSON body. The JSON body must use MCP methods such as `initialize`, `tools/list`, and `tools/call`.

In [ ]:
import json

initialize_body = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {
            "name": "power-platform-test",
            "version": "1.0.0"
        }
    }
}

tools_list_body = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list",
    "params": {}
}

tools_call_template = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "<tool-name-from-tools-list>",
        "arguments": {}
    }
}

print("POWER PLATFORM TEST TAB PAYLOADS")
print("")
print("Before testing, confirm these Security values for direct Fabric testing:")
print("- Resource URL: https://api.fabric.microsoft.com")
print("- Scope: https://api.fabric.microsoft.com/.default")
print("")
print("1. Raw Body for initialize")
print(json.dumps(initialize_body, indent=2))
print("")
print("2. Raw Body for tools/list")
print(json.dumps(tools_list_body, indent=2))
print("")
print("3. Raw Body template for tools/call")
print(json.dumps(tools_call_template, indent=2))
print("")
print("Use one of the tool names returned by tools/list, then replace the arguments object with the values that tool expects.")